# Занятие 2. Лабораторная: эксперименты с линейной регрессией

Лабораторная целиком домашняя. В ней семь задач, все обязательные, бонусов нет.

Половина задач — реализовать что-то руками. Вторая половина — провести
эксперимент и **сделать выбор по метрике**: какой шаг обучения, какая степень
полинома, какая регуляризация, какая функция потерь. Проверка пересчитывает
каждый эксперимент независимо, поэтому засчитывается только тот выбор,
который действительно подтверждается числами.

Все, что нужно для решения, написано в теории перед задачами. Читайте ее
внимательно: подсказок в самих заготовках нет.

Имена переменных и функций в заготовках менять нельзя — проверка ищет
их по именам. После каждой задачи идет ячейка с открытыми проверками;
при сдаче работа дополнительно проверяется закрытыми тестами на других данных.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso, LinearRegression, QuantileRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.seterr(over="ignore", invalid="ignore")

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 42

## Задача 1. Метрики регрессии

Реализуйте четыре метрики на numpy, без `sklearn.metrics`. Функции принимают
массивы или списки одинаковой длины и возвращают число.

$$\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
\qquad
\mathrm{RMSE} = \sqrt{\mathrm{MSE}}
\qquad
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

Здесь $\bar{y}$ — среднее **истинных** значений той же выборки, на которой
считается метрика. Первым аргументом во всех функциях идут истинные значения,
вторым — предсказанные.

In [ ]:
def mse(y_true, y_pred):
    return ...


def rmse(y_true, y_pred):
    return ...


def mae(y_true, y_pred):
    return ...


def r2(y_true, y_pred):
    return ...

In [ ]:
# --- проверка ---
rng_check = np.random.default_rng(0)
a = rng_check.normal(size=50)
b = a + rng_check.normal(scale=0.5, size=50)

# 1. совпадают с библиотекой
assert np.isclose(mse(a, b), mean_squared_error(a, b)), "MSE не совпала со sklearn"
assert np.isclose(mae(a, b), mean_absolute_error(a, b)), "MAE не совпала со sklearn"
assert np.isclose(r2(a, b), r2_score(a, b)), "R2 не совпал со sklearn"

# 2. RMSE — корень из MSE
assert np.isclose(rmse(a, b), np.sqrt(mse(a, b))), "RMSE не равен корню из MSE"

# 3. идеальное предсказание
assert mse(a, a) == 0 and np.isclose(r2(a, a), 1.0), "на идеальном предсказании MSE 0 и R2 1"

# 4. предсказание средним дает R2 ровно ноль
assert np.isclose(r2(a, np.full_like(a, a.mean())), 0.0), "R2 константы-среднего должен быть 0"

# 5. списки тоже принимаются
assert np.isclose(mse([1, 2, 3], [1, 2, 4]), 1 / 3), "функции должны принимать списки"

print("проверки пройдены")

## Задача 2. Нормальное уравнение и Ridge в явном виде

Данные — вино из семинара: предсказываем крепость по остальным двенадцати
признакам. Разбиение готово.

### Обычная регрессия

Модель $\hat{y} = Xw + b$. Чтобы найти сдвиг вместе с весами, к таблице
приписывают слева столбец единиц: $A = [\mathbf{1}, X]$. Тогда
$\hat{y} = A\theta$, где $\theta = (b, w_1, \dots, w_d)$, и

$$\theta = (A^\top A)^{-1} A^\top y$$

Обращать матрицу явно не нужно: систему $A^\top A\,\theta = A^\top y$ решает
`np.linalg.solve`, это точнее. Функция `fit_linear(X, y)` возвращает кортеж
`(b, w)`: число и вектор длины $d$.

### Ridge

Штраф $\alpha\|w\|^2$ накладывается только на веса, сдвиг не штрафуется,
иначе результат зависел бы от того, где на оси лежат значения $y$. Стандартный
способ этого добиться — центрировать данные:

$$X_c = X - \bar{x}, \qquad y_c = y - \bar{y}$$

где $\bar{x}$ — вектор средних по столбцам. На центрированных данных сдвиг
равен нулю, и веса находятся явно:

$$w = (X_c^\top X_c + \alpha I)^{-1} X_c^\top y_c,
\qquad
b = \bar{y} - \bar{x}^\top w$$

Именно так устроен `Ridge` в sklearn. Функция `fit_ridge(X, y, alpha)` тоже
возвращает `(b, w)`.

In [ ]:
wine = load_wine(as_frame=True)
data = wine.frame.drop(columns="target")

y = data["alcohol"].to_numpy()
features = data.drop(columns="alcohol")
feature_names = list(features.columns)
X = features.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)
print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
def fit_linear(X, y):
    A = ...
    theta = ...
    return ..., ...


def fit_ridge(X, y, alpha):
    x_mean = ...
    y_mean = ...
    X_c = ...
    y_c = ...
    w = ...
    b = ...
    return b, w

In [ ]:
# --- проверка ---
reference = LinearRegression().fit(X_train, y_train)
b_lin, w_lin = fit_linear(X_train, y_train)

# 1. обычная регрессия совпадает со sklearn
assert np.isclose(b_lin, reference.intercept_), "сдвиг не совпал с LinearRegression"
assert np.allclose(w_lin, reference.coef_), "веса не совпали с LinearRegression"

# 2. Ridge совпадает со sklearn при разной силе штрафа
for alpha in (1.0, 30.0):
    b_r, w_r = fit_ridge(X_train, y_train, alpha)
    ref = Ridge(alpha=alpha).fit(X_train, y_train)
    assert np.isclose(b_r, ref.intercept_), f"сдвиг Ridge не совпал при alpha={alpha}"
    assert np.allclose(w_r, ref.coef_), f"веса Ridge не совпали при alpha={alpha}"

# 3. при нулевом штрафе Ridge превращается в обычную регрессию
b0, w0 = fit_ridge(X_train, y_train, 0.0)
assert np.allclose(w0, w_lin) and np.isclose(b0, b_lin), "Ridge с alpha=0 должен совпасть с fit_linear"

# 4. формы
assert np.ndim(b_lin) == 0 and w_lin.shape == (X_train.shape[1],), "сдвиг — число, веса — вектор длины d"

print("проверки пройдены")

## Задача 3. Градиентный спуск и выбор шага

### Алгоритм

Работаем с матрицей $A$, в которой уже есть столбец единиц, и ищем вектор
$\theta$. Функция потерь — среднеквадратичная ошибка:

$$L(\theta) = \frac{1}{n}\|y - A\theta\|^2,
\qquad
\nabla L(\theta) = -\frac{2}{n} A^\top (y - A\theta)$$

Спуск начинается с нулевого вектора и `n_iter` раз делает шаг
$\theta \leftarrow \theta - \eta\,\nabla L(\theta)$. После **каждого** шага значение
$L(\theta)$ дописывается в список `history`. Функция
`gradient_descent(A, y, lr, n_iter)` возвращает `(theta, history)`, длина
`history` равна `n_iter`.

Если шаг слишком большой, значения потерь растут и превращаются в `inf`
или `nan`. Это нормальный исход эксперимента, прерывать спуск не нужно.

### Эксперимент

Матрица `A_train` и минимально возможные потери `optimal_loss` посчитаны
ниже. Для каждого шага из `learning_rates` запустите спуск на `N_ITER`
итераций и найдите номер первой итерации, на которой потери отличаются
от оптимальных меньше чем на `TOLERANCE`. Нумерация итераций с единицы:
потери после первого шага — это итерация 1.

Сохраните результаты в словарь `iterations_needed`: ключ — шаг, значение —
номер итерации или `None`, если за `N_ITER` итераций точность не достигнута
или спуск разошелся. В `best_lr` положите шаг, которому потребовалось меньше
всего итераций.

### Почему масштаб признаков важен

Для квадратичной функции потерь спуск устойчив, только если шаг меньше
$2 / \lambda_{\max}$, где $\lambda_{\max}$ — наибольшее собственное число
матрицы $\frac{2}{n}A^\top A$. Крупный по масштабу признак раздувает это число,
и допустимый шаг становится крошечным. Поэтому матрица `A_train` построена
из стандартизованных признаков.

In [ ]:
scaler_gd = StandardScaler().fit(X_train)
# столбец единиц для свободного члена
A_train = np.column_stack([np.ones(len(X_train)), scaler_gd.transform(X_train)])

theta_best = np.linalg.lstsq(A_train, y_train, rcond=None)[0]
optimal_loss = np.mean((A_train @ theta_best - y_train) ** 2)

N_ITER = 3000
TOLERANCE = 1e-4
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3]

print(f"минимальные потери: {optimal_loss:.5f}")

In [ ]:
def gradient_descent(A, y, lr, n_iter):
    theta = ...
    history = ...
    for _ in range(n_iter):
        ...
    return theta, history


iterations_needed = ...
best_lr = ...

for lr, count in iterations_needed.items():
    print(f"  шаг {lr:<6} итераций до точности: {count}")
print("лучший шаг:", best_lr)

In [ ]:
# --- проверка ---

# 1. история нужной длины
theta_gd, history_gd = gradient_descent(A_train, y_train, 0.1, 200)
assert len(history_gd) == 200, "длина history должна равняться n_iter"

# 2. при хорошем шаге спуск приходит к точному решению
theta_long, _ = gradient_descent(A_train, y_train, 0.1, N_ITER)
assert np.allclose(theta_long, theta_best, atol=1e-4), "спуск не пришел к решению нормального уравнения"

# 3. при хорошем шаге потери не растут
assert all(later <= earlier + 1e-12 for earlier, later in zip(history_gd, history_gd[1:])), (
    "при устойчивом шаге потери не должны расти"
)

# 4. словарь заполнен для всех шагов
assert set(iterations_needed) == set(learning_rates), "iterations_needed должен содержать все шаги"

# 5. выбранный шаг действительно самый быстрый из сошедшихся
finished = {lr: count for lr, count in iterations_needed.items() if count is not None}
assert best_lr == min(finished, key=finished.get), "best_lr не соответствует минимуму итераций"

print("проверки пройдены")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for lr, color in zip(learning_rates, ["#9CA3AF", "#6B7280", "#4DA3E0", "#0072CE", "#E03A3A"]):
    _, history = gradient_descent(A_train, y_train, lr, 300)
    losses = np.clip(np.array(history, dtype=float), None, 10)
    ax.plot(np.arange(1, 301), losses - optimal_loss + 1e-8, color=color, label=f"шаг {lr}")
ax.set_yscale("log")
ax.set_xlabel("итерация")
ax.set_ylabel("потери минус оптимум")
ax.set_title("Первые 300 итераций при разных шагах")
ax.legend()
plt.show()

A_raw = np.column_stack([np.ones(len(X_train)), X_train])
limit_scaled = 2 / np.linalg.eigvalsh(2 * A_train.T @ A_train / len(A_train)).max()
limit_raw = 2 / np.linalg.eigvalsh(2 * A_raw.T @ A_raw / len(A_raw)).max()
print(f"допустимый шаг со стандартизацией:  меньше {limit_scaled:.3f}")
print(f"допустимый шаг без стандартизации: меньше {limit_raw:.2e}")

**Вопрос.** Сравните допустимый шаг со стандартизацией и без нее. Какой признак
отвечает за такую разницу и почему спуск без стандартизации практически
не сдвинется по остальным признакам даже при допустимом шаге?

*Ответ пишите здесь.*

## Задача 4. Функция потерь и выбросы

Выбор функции потерь — это предположение о шуме. Квадрат ошибки соответствует
нормальному шуму: вероятность большой ошибки убывает очень быстро, и модель
изо всех сил старается не допустить ни одной. Поэтому единичные выбросы
тянут решение на себя.

Модуль ошибки соответствует шуму с распределением Лапласа, у которого хвосты
тяжелее. Такая модель спокойнее относится к редким большим отклонениям.
В sklearn регрессию с функцией потерь MAE строит
`QuantileRegressor(quantile=0.5, alpha=0, solver="highs")`: при квантиле 0.5
он минимизирует сумму модулей остатков.

Данные ниже: прямая $y = 2x + 1$ с шумом, в десяти процентах обучающих
объектов к ответу прибавлено 25. Тестовая выборка чистая.

Обучите на `x_train_out, y_train_out` две модели: `model_mse` —
`LinearRegression`, `model_mae` — `QuantileRegressor` с параметрами выше.
Посчитайте MAE каждой на тестовой выборке и сохраните в `test_mae_mse`
и `test_mae_mae`. В `robust_choice` запишите строку `"mse"` или `"mae"` —
какая модель ошиблась на тесте меньше.

In [ ]:
rng_out = np.random.default_rng(SEED)
x_all = rng_out.uniform(0, 10, 200)
y_all = 2 * x_all + 1 + rng_out.normal(0, 1, 200)

x_train_out, x_test_out = x_all[:140, None], x_all[140:, None]
y_train_out, y_test_out = y_all[:140].copy(), y_all[140:]
corrupted = rng_out.choice(140, 14, replace=False)
y_train_out[corrupted] += 25

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_train_out, y_train_out, s=20, color="#0072CE", alpha=0.7, label="обучающие")
ax.scatter(x_train_out[corrupted], y_train_out[corrupted], s=40, color="#E03A3A", label="выбросы")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

In [ ]:
model_mse = ...
model_mae = ...

test_mae_mse = ...
test_mae_mae = ...

robust_choice = ...

print(f"MSE-модель: наклон {model_mse.coef_[0]:.2f}, MAE на тесте {test_mae_mse:.3f}")
print(f"MAE-модель: наклон {model_mae.coef_[0]:.2f}, MAE на тесте {test_mae_mae:.3f}")
print("выбор:", robust_choice)

In [ ]:
# --- проверка ---

# 1. обе модели обучены
assert hasattr(model_mse, "coef_") and hasattr(model_mae, "coef_"), "обе модели должны быть обучены"

# 2. метрики пересчитываются
assert np.isclose(test_mae_mse, mean_absolute_error(y_test_out, model_mse.predict(x_test_out)))
assert np.isclose(test_mae_mae, mean_absolute_error(y_test_out, model_mae.predict(x_test_out)))

# 3. выбор соответствует числам
expected_choice = "mse" if test_mae_mse < test_mae_mae else "mae"
assert robust_choice == expected_choice, "robust_choice не соответствует посчитанным MAE"

# 4. устойчивая модель восстановила наклон ближе к настоящему
assert abs(model_mae.coef_[0] - 2) < abs(model_mse.coef_[0] - 2), (
    "наклон MAE-модели должен быть ближе к настоящему значению 2"
)

print("проверки пройдены")

In [ ]:
grid_out = np.linspace(0, 10, 50)[:, None]
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_train_out, y_train_out, s=18, color="#9CA3AF", alpha=0.7)
ax.plot(grid_out, 2 * grid_out + 1, color="black", ls="--", label="настоящая прямая")
ax.plot(grid_out, model_mse.predict(grid_out), color="#E03A3A", lw=2, label="MSE")
ax.plot(grid_out, model_mae.predict(grid_out), color="#0072CE", lw=2, label="MAE")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

**Вопрос.** Почему модель с квадратичной функцией потерь сдвинулась вверх сильнее,
чем изменила наклон? Подумайте, где на оси $x$ оказались выбросы.

*Ответ пишите здесь.*

## Задача 5. Сложность модели и переобучение

Линейная регрессия умеет строить не только прямые. Если добавить признаки
$x^2, x^3, \dots, x^d$, модель останется линейной по весам, но сможет описывать
кривые. В sklearn такие признаки строит `PolynomialFeatures(d)`.

Чем больше степень, тем гибче модель. На обучающих данных ошибка с ростом
степени только падает: у более гибкой модели больше возможностей подогнать
каждую точку. На высоких степенях это правило может нарушаться на десятитысячные
доли: признаки вида $x^{12}$ на отрезке от нуля до единицы очень малы,
и решатель теряет точность вычислений. На новых данных она сначала падает, а потом растет — модель
начинает выучивать шум. Поэтому степень, как любой гиперпараметр, выбирают
по данным, которых модель не видела, а не по обучающим.

Данные ниже: 40 точек синусоиды с шумом. Для каждой степени из `degrees`
соберите конвейер `make_pipeline(PolynomialFeatures(d), LinearRegression())`
и посчитайте две величины:

- `train_rmse` — RMSE на всех 40 точках после обучения на всех 40 точках
- `cv_rmse` — RMSE на кросс-валидации с готовым разбиением `cv_poly`,
  усредненный по фолдам

Оба — списки в порядке `degrees`. В `best_degree` положите степень с наименьшим
`cv_rmse`.

In [ ]:
rng_poly = np.random.default_rng(8)
x_poly = rng_poly.uniform(0, 1, 40)
y_poly = np.sin(2 * np.pi * x_poly) + rng_poly.normal(0, 0.2, 40)
X_poly = x_poly[:, None]

degrees = list(range(1, 13))
cv_poly = KFold(n_splits=5, shuffle=True, random_state=SEED)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(x_poly, y_poly, color="#0072CE")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

In [ ]:
train_rmse = ...
cv_rmse = ...
best_degree = ...

print("лучшая степень:", best_degree)

In [ ]:
# --- проверка ---

# 1. по значению на каждую степень
assert len(train_rmse) == len(degrees) == len(cv_rmse), "длины списков не совпадают со степенями"

# 2. на обучающих данных ошибка не растет с ростом степени, с точностью до вычислений
assert all(later <= earlier + 1e-3 for earlier, later in zip(train_rmse, train_rmse[1:])), (
    "train_rmse заметно растет с ростом степени, так быть не должно"
)
assert train_rmse[-1] < train_rmse[0] / 2, "на высокой степени ошибка на обучении должна быть заметно ниже"


# 3. кросс-валидация пересчитывается для одной из степеней
check = make_pipeline(PolynomialFeatures(3), LinearRegression())
expected = -cross_val_score(check, X_poly, y_poly, cv=cv_poly, scoring="neg_root_mean_squared_error").mean()
assert np.isclose(cv_rmse[2], expected), "cv_rmse для степени 3 не совпал с пересчетом"

# 4. выбор сделан по кросс-валидации
assert best_degree == degrees[int(np.argmin(cv_rmse))], "best_degree не соответствует минимуму cv_rmse"

# 5. и это не самая сложная модель
assert best_degree < max(degrees), "лучшей по кросс-валидации не должна быть самая высокая степень"

print("проверки пройдены")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(degrees, train_rmse, "o-", color="#6B7280", label="на обучении")
axes[0].plot(degrees, cv_rmse, "o-", color="#0072CE", label="на кросс-валидации")
axes[0].set_yscale("log")
axes[0].set_xlabel("степень полинома")
axes[0].set_ylabel("RMSE")
axes[0].legend()

grid_poly = np.linspace(0, 1, 300)[:, None]
axes[1].scatter(x_poly, y_poly, color="#9CA3AF")
for d, color in [(1, "#6B7280"), (best_degree, "#0072CE"), (12, "#E03A3A")]:
    fitted = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_poly, y_poly)
    axes[1].plot(grid_poly, fitted.predict(grid_poly), color=color, lw=2, label=f"степень {d}")
axes[1].set_ylim(-2, 2)
axes[1].legend()
plt.tight_layout()
plt.show()

**Вопрос.** Какой была бы «лучшая» степень, если выбирать ее по `train_rmse`?
Что при этом случилось бы с ошибкой на новых данных?

*Ответ пишите здесь.*

## Задача 6. Ridge или Lasso

Обе регрессии штрафуют веса, но по-разному. L2-штраф соответствует нормальному
априорному распределению: веса скорее маленькие, но все ненулевые. L1-штраф
соответствует распределению Лапласа, у которого острый пик в нуле: многие
веса скорее всего равны нулю. Поэтому Lasso обнуляет часть весов, а Ridge
только уменьшает их.

Отсюда следует, что лучший штраф зависит от того, как устроены данные. Если
на ответ влияют все признаки понемногу, ближе к правде предположение Ridge.
Если влияют лишь несколько признаков, а остальные — шум, ближе предположение
Lasso.

Ниже два набора данных одинакового размера: 60 объектов, 60 признаков.
В `dense` на ответ влияют все признаки, в `sparse` — только пять.
Истинные веса вам известны не будут, выбор нужно сделать по метрике.

Для каждого набора посчитайте RMSE на кросс-валидации с разбиением `cv_reg`
для всех кандидатов из `candidates` и сохраните в словари `scores_dense`
и `scores_sparse`. Ключ словаря — кортеж из `candidates`, например `("ridge", 1)`,
значение — RMSE, усредненный по фолдам. Модель для кандидата собирается так:

- `("ols", None)` — `make_pipeline(StandardScaler(), LinearRegression())`
- `("ridge", a)` — `make_pipeline(StandardScaler(), Ridge(alpha=a))`
- `("lasso", a)` — `make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=50000))`

В `choice_dense` и `choice_sparse` положите кандидата с наименьшим RMSE.

In [ ]:
def make_correlated(n, d, rho, rng):
    noise = rng.normal(size=(n, d))
    X_corr = np.empty((n, d))
    X_corr[:, 0] = noise[:, 0]
    for j in range(1, d):
        X_corr[:, j] = rho * X_corr[:, j - 1] + np.sqrt(1 - rho ** 2) * noise[:, j]
    return X_corr


rng_dense = np.random.default_rng(8)
X_dense = make_correlated(60, 60, 0.7, rng_dense)
y_dense = X_dense @ rng_dense.normal(0, 1, 60) + rng_dense.normal(0, 2, 60)

rng_sparse = np.random.default_rng(101)
X_sparse = rng_sparse.normal(size=(60, 60))
true_sparse = np.zeros(60)
true_sparse[:5] = 3.0
y_sparse = X_sparse @ true_sparse + rng_sparse.normal(0, 1, 60)

cv_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
candidates = (
    [("ols", None)]
    + [("ridge", a) for a in [0.1, 1, 10, 100, 1000]]
    + [("lasso", a) for a in [0.001, 0.01, 0.1, 0.5, 1]]
)
print("кандидатов:", len(candidates))

In [ ]:
scores_dense = ...
scores_sparse = ...

choice_dense = ...
choice_sparse = ...

print("dense: ", choice_dense, round(scores_dense[choice_dense], 3))
print("sparse:", choice_sparse, round(scores_sparse[choice_sparse], 3))

In [ ]:
# --- проверка ---

# 1. посчитаны все кандидаты
assert set(scores_dense) == set(candidates) and set(scores_sparse) == set(candidates), (
    "в словарях должны быть все кандидаты из candidates"
)

# 2. значения — RMSE, а не отрицательные числа
assert all(v > 0 for v in list(scores_dense.values()) + list(scores_sparse.values())), (
    "RMSE должен быть положительным"
)

# 3. одно значение пересчитывается
check = -cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=10)), X_dense, y_dense,
                         cv=cv_reg, scoring="neg_root_mean_squared_error").mean()
assert np.isclose(scores_dense[("ridge", 10)], check), "scores_dense для ridge 10 не совпал с пересчетом"

# 4. выбор соответствует минимуму
assert choice_dense == min(scores_dense, key=scores_dense.get), "choice_dense не соответствует минимуму"
assert choice_sparse == min(scores_sparse, key=scores_sparse.get), "choice_sparse не соответствует минимуму"

# 5. на двух наборах победили разные типы штрафа
assert choice_dense[0] != choice_sparse[0], "на двух наборах должны выиграть разные типы регуляризации"

print("проверки пройдены")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, scores, title in [(axes[0], scores_dense, "dense"), (axes[1], scores_sparse, "sparse")]:
    labels = [f"{kind}\n{alpha}" if alpha is not None else kind for kind, alpha in scores]
    colors = ["#6B7280" if kind == "ols" else "#0072CE" if kind == "ridge" else "#C98A3C"
              for kind, _ in scores]
    ax.bar(range(len(scores)), list(scores.values()), color=colors)
    ax.set_xticks(range(len(scores)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("RMSE на кросс-валидации")
    ax.set_title(title)
plt.tight_layout()
plt.show()

**Вопрос.** Обучите Lasso с выбранным `alpha` на наборе `sparse` и посчитайте,
сколько весов он оставил ненулевыми. Сходится ли это с тем, что сказано
в условии про устройство данных?

*Ответ пишите здесь.*

## Задача 7. Мультиколлинеарность на вине

Когда признаки почти линейно связаны, веса линейной регрессии становятся
неустойчивыми: небольшое изменение данных сильно меняет их, иногда вплоть
до смены знака. Проверить это можно бутстрэпом: много раз обучить модель
на выборках, составленных из обучающих объектов с повторениями, и посмотреть
на разброс весов.

Используем вино из второй задачи. Признаки стандартизованы (`X_train_std`),
индексы двухсот бутстрэп-выборок готовы: `boot_idx[k]` — номера объектов
для $k$-го обучения.

Посчитайте:

- `ols_weights` — таблица `pd.DataFrame` размера 200 на 12: строка $k$ — веса
  `LinearRegression`, обученной на `X_train_std[boot_idx[k]]`, `y_train[boot_idx[k]]`;
  столбцы названы по `feature_names`
- `ridge_weights` — то же для `Ridge(alpha=30)`
- `most_unstable` — имя признака с наибольшим стандартным отклонением веса
  в `ols_weights`
- `spread_ratio` — во сколько раз стандартное отклонение веса этого признака
  в `ols_weights` больше, чем в `ridge_weights`
- `sign_flip_share` — доля строк `ols_weights`, в которых знак веса этого
  признака отличается от знака его медианы

Стандартное отклонение считайте методом `pandas` по умолчанию.

In [ ]:
X_train_std = StandardScaler().fit_transform(X_train)
boot_idx = np.random.default_rng(SEED).integers(0, len(y_train), size=(200, len(y_train)))
print("бутстрэп-выборок:", boot_idx.shape[0], " объектов в каждой:", boot_idx.shape[1])

In [ ]:
ols_weights = ...
ridge_weights = ...

most_unstable = ...
spread_ratio = ...
sign_flip_share = ...

print(f"самый нестабильный вес: {most_unstable}")
print(f"Ridge уменьшает его разброс в {spread_ratio:.2f} раза")
print(f"знак меняется в {sign_flip_share:.0%} обучений")

In [ ]:
# --- проверка ---

# 1. таблицы нужной формы с именами признаков
assert ols_weights.shape == (200, 12) and ridge_weights.shape == (200, 12), "таблицы должны быть 200 на 12"
assert list(ols_weights.columns) == feature_names, "столбцы должны называться по feature_names"

# 2. первая строка — веса модели на первой бутстрэп-выборке
first = LinearRegression().fit(X_train_std[boot_idx[0]], y_train[boot_idx[0]]).coef_
assert np.allclose(ols_weights.iloc[0].to_numpy(), first), "первая строка ols_weights не совпала с пересчетом"

# 3. самый нестабильный признак выбран по данным
assert most_unstable == ols_weights.std().idxmax(), "most_unstable не соответствует максимуму разброса"

# 4. отношение разбросов пересчитывается и больше единицы
expected_ratio = ols_weights[most_unstable].std() / ridge_weights[most_unstable].std()
assert np.isclose(spread_ratio, expected_ratio) and spread_ratio > 1, "spread_ratio посчитан неверно"

# 5. доля смены знака — число от нуля до одной второй
assert 0 <= sign_flip_share <= 0.5, "доля смены знака относительно медианы не может превышать половину"

print("проверки пройдены")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ols_weights[most_unstable], bins=30, alpha=0.6, color="#0072CE", label="без регуляризации")
ax.hist(ridge_weights[most_unstable], bins=30, alpha=0.7, color="#E03A3A", label="Ridge, alpha=30")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel(f"вес признака {most_unstable}")
ax.set_ylabel("сколько обучений")
ax.legend()
plt.show()

pair_corr = np.corrcoef(X_train_std, rowvar=False)
partner = feature_names[int(np.argsort(np.abs(pair_corr[feature_names.index(most_unstable)]))[-2])]
print(f"сильнее всего {most_unstable} связан с признаком {partner}")

**Вопрос.** Посмотрите в словарь курса, `terminology/glossary.md`, что химически
означают `most_unstable` и найденный напарник. Объясните, почему модели все равно,
как поделить вклад между ними, и почему предсказания при этом почти не меняются.

*Ответ пишите здесь.*

## Итог

После лабораторной вы должны уметь объяснить:

- откуда в линейной регрессии берется формула для весов и почему у Ridge сдвиг
  не штрафуется
- почему градиентный спуск без стандартизации практически не работает
- почему функция потерь — это предположение о шуме
- почему гиперпараметр нельзя выбирать по ошибке на обучающих данных
- от чего зависит, какая регуляризация лучше
- почему при мультиколлинеарности нельзя верить отдельным весам